In [ ]:
from scipy.io import loadmat
import pandas as pd
import numpy as np

Read mat file

In [2]:
# Load the .mat file
elec_data = loadmat('raw_data/dataset_elec.mat')
amb_data = loadmat('raw_data/dataset_amb.mat')

# Inspect keys (MATLAB variables stored inside)
print(elec_data.keys())
print(amb_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'idc1', 'idc2', 'vdc1', 'vdc2'])
dict_keys(['__header__', '__version__', '__globals__', 'f_nv', 'irr', 'pvt'])


Convert to Dataframe

In [3]:
df = pd.DataFrame({
    'idc1': elec_data['idc1'].flatten(),
    'idc2': elec_data['idc2'].flatten(),
    'vdc1': elec_data['vdc1'].flatten(),
    'vdc2': elec_data['vdc2'].flatten(),
    'irr': amb_data['irr'].flatten(),
    'pvt': amb_data['pvt'].flatten(),
    'f_nv':amb_data['f_nv'].flatten()
})

df['f_nv'] = df['f_nv'].astype('category')

print(df.shape)
print(df.head())

(1373798, 7)
     idc1    idc2    vdc1    vdc2     irr     pvt f_nv
0  0.0608  0.0073  0.7143  0.5550  1.3729  2.3816    0
1  0.0615  0.0064  0.6944  0.5427  1.3604  2.3816    0
2  0.0646  0.0067  0.7110  0.5583  1.5118  2.3883    0
3  0.0628  0.0067  0.6991  0.5465  1.5534  2.3920    0
4  0.0606  0.0076  0.7035  0.5553  1.4355  2.3920    0


In [4]:
print(df.dtypes)

idc1     float64
idc2     float64
vdc1     float64
vdc2     float64
irr      float64
pvt      float64
f_nv    category
dtype: object


Check Missing Values

In [5]:
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)

      Missing Count  Missing %
idc1              0        0.0
idc2              0        0.0
vdc1              0        0.0
vdc2              0        0.0
irr               0        0.0
pvt               0        0.0
f_nv              0        0.0


Descriptive Statistics

In [6]:
# Display describe() without scientific notation
with pd.option_context('display.float_format', '{:,.6f}'.format):
    print(df.describe())

                  idc1             idc2             vdc1             vdc2  \
count 1,373,798.000000 1,373,798.000000 1,373,798.000000 1,373,798.000000   
mean          2.276919         2.000344       135.118273       134.719897   
std           3.190345         3.038147       136.333921       136.730871   
min           0.041800         0.004600         0.379900         0.316400   
25%           0.068600         0.012700         0.890700         0.884000   
50%           0.113600         0.044100        37.776250        39.837450   
75%           5.102100         4.140400       273.060000       271.986000   
max           9.654900         9.498700       363.974000       368.749000   

                   irr              pvt  
count 1,373,798.000000 1,373,798.000000  
mean        257.611072        22.772770  
std         353.121870        14.231485  
min          -0.038100        -1.654300  
25%           1.290100        12.132700  
50%           1.736400        18.490450  
75%         

Outlier Detection

In [7]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Outlier Count'])
print(outlier_df)

      Outlier Count
idc1              0
idc2              0
vdc1              0
vdc2              0
irr               0
pvt               0


Save Dataframe

In [8]:
df.to_csv("data/solar_string_data.csv", index=False)